In [ ]:
import os
import gc
import nibabel as nib
import numpy as np
import cv2
from tqdm import tqdm

# ----------------------------------------
# DATASET PATH
# ----------------------------------------

dataset_path = "../dataset/BraTs2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/"

# ----------------------------------------
# PARAMETERS
# ----------------------------------------

IMG_SIZE = 128
SLICE_START = 30
SLICE_END = 120

# ----------------------------------------
# PROCESS DATA IN SMALL CHUNKS
# ----------------------------------------
# CHANGE:
# Instead of processing everything together,
# process smaller patient ranges to avoid RAM crash.

START_PATIENT = 0
END_PATIENT = 30

# ----------------------------------------
# OUTPUT FILE NAMES
# ----------------------------------------
# CHANGE:
# Save processed data directly to disk.

X_SAVE_PATH = f"X_{START_PATIENT}_{END_PATIENT}.npy"
Y_SAVE_PATH = f"Y_{START_PATIENT}_{END_PATIENT}.npy"

# ----------------------------------------
# FINAL DATASET ARRAYS
# ----------------------------------------

X = []
Y = []

# ----------------------------------------
# GET PATIENT LIST
# ----------------------------------------

patients = os.listdir(dataset_path)

# CHANGE:
# Process only selected chunk.

patients = patients[START_PATIENT:END_PATIENT]

print("Total patients being processed:", len(patients))

# ----------------------------------------
# NORMALIZATION FUNCTION
# ----------------------------------------
# CHANGE:
# Moved outside loop for efficiency.

def normalize(image):
    if np.max(image) == 0:
        return image
    return image / np.max(image)

# ----------------------------------------
# LOOP THROUGH PATIENTS
# ----------------------------------------

for patient in tqdm(patients):

    patient_path = os.path.join(dataset_path, patient)

    # File paths

    flair_path = os.path.join(patient_path, f"{patient}_flair.nii")
    t1_path = os.path.join(patient_path, f"{patient}_t1.nii")
    t1ce_path = os.path.join(patient_path, f"{patient}_t1ce.nii")
    t2_path = os.path.join(patient_path, f"{patient}_t2.nii")
    seg_path = os.path.join(patient_path, f"{patient}_seg.nii")

    # ----------------------------------------
    # LOAD SCANS
    # ----------------------------------------

    flair = nib.load(flair_path).get_fdata()
    t1 = nib.load(t1_path).get_fdata()
    t1ce = nib.load(t1ce_path).get_fdata()
    t2 = nib.load(t2_path).get_fdata()
    seg = nib.load(seg_path).get_fdata()

    # ----------------------------------------
    # LOOP THROUGH SLICES
    # ----------------------------------------

    for slice_num in range(SLICE_START, SLICE_END):

        # Extract slices

        flair_slice = flair[:, :, slice_num]
        t1_slice = t1[:, :, slice_num]
        t1ce_slice = t1ce[:, :, slice_num]
        t2_slice = t2[:, :, slice_num]
        seg_slice = seg[:, :, slice_num]

        # ----------------------------------------
        # RESIZE MRI IMAGES
        # ----------------------------------------

        flair_slice = cv2.resize(flair_slice, (IMG_SIZE, IMG_SIZE))
        t1_slice = cv2.resize(t1_slice, (IMG_SIZE, IMG_SIZE))
        t1ce_slice = cv2.resize(t1ce_slice, (IMG_SIZE, IMG_SIZE))
        t2_slice = cv2.resize(t2_slice, (IMG_SIZE, IMG_SIZE))

        # ----------------------------------------
        # RESIZE MASK
        # ----------------------------------------
        # CHANGE:
        # INTER_NEAREST preserves mask labels correctly.

        seg_slice = cv2.resize(
            seg_slice,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_NEAREST
        )

        # ----------------------------------------
        # NORMALIZATION
        # ----------------------------------------

        flair_slice = normalize(flair_slice)
        t1_slice = normalize(t1_slice)
        t1ce_slice = normalize(t1ce_slice)
        t2_slice = normalize(t2_slice)

        # ----------------------------------------
        # STACK MODALITIES
        # ----------------------------------------

        combined_image = np.stack(
            [flair_slice, t1_slice, t1ce_slice, t2_slice],
            axis=-1
        )

        # ----------------------------------------
        # BINARY MASK
        # ----------------------------------------

        seg_binary = np.where(seg_slice > 0, 1, 0)

        # Add channel dimension

        seg_binary = np.expand_dims(seg_binary, axis=-1)

        # ----------------------------------------
        # STORE DATA
        # ----------------------------------------

        X.append(combined_image.astype(np.float32))
        Y.append(seg_binary.astype(np.uint8))

    # ----------------------------------------
    # FREE MEMORY AFTER EACH PATIENT
    # ----------------------------------------
    # CHANGE:
    # Prevent memory buildup.

    del flair
    del t1
    del t1ce
    del t2
    del seg

    gc.collect()

# ----------------------------------------
# CHECK TOTAL SAMPLES
# ----------------------------------------

print("Total slices collected:", len(X))

# ----------------------------------------
# CONVERT TO NUMPY ARRAYS
# ----------------------------------------
# CHANGE:
# This is where crashes usually happen,
# but smaller chunks make it safer.

X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.uint8)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

# ----------------------------------------
# SAVE DATASET TO DISK
# ----------------------------------------
# CHANGE:
# Save immediately so work is not lost.

np.save(X_SAVE_PATH, X)
np.save(Y_SAVE_PATH, Y)

print(f"\nSaved X to: {X_SAVE_PATH}")
print(f"Saved Y to: {Y_SAVE_PATH}")

# ----------------------------------------
# OPTIONAL MEMORY CLEANUP
# ----------------------------------------

del X
del Y

gc.collect()

print("\nProcessing completed successfully!")